# Calling an LLM from Python
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/engr1050-fall2026/blob/main/NewMaterial/LLM_API/llm_api.ipynb)

## Overview

So far we have used Gemini through the chat panel built into Colab. A Python program can use the same model directly: it sends a prompt to Google's servers as a string and gets the model's reply back as another string. This notebook does that in about five lines of code.

Before running anything, follow the [setup instructions](https://natrask.github.io/engr1050-fall2026/NewMaterial/LLM_API/) to create a free Gemini API key and store it in Colab's Secrets panel under the name `GEMINI_API_KEY`.

**Reminder:** on the free tier, Google uses your prompts and the model's replies to improve its products, and human reviewers may read them. Do not type passwords, personal information, or anything about other students.

## Step 1: Connect to Gemini

`google-genai` is Google's Python package for the Gemini API, and Colab already has it installed. The cell below reads your key from the Secrets panel and builds a `client`, an object whose methods send requests to Gemini. `MODEL` names the model that will answer; `gemini-3.1-flash-lite` is free and fast.

If Colab asks whether to grant this notebook access to your secret, click **Grant access**.

In [ ]:
from google import genai
from google.colab import userdata

# Read your key from Colab's Secrets panel (the key icon on the left)
API_KEY = userdata.get('GEMINI_API_KEY')

# The client sends your prompts to Google and brings back the replies
client = genai.Client(api_key=API_KEY)

# Which Gemini model answers
MODEL = 'gemini-3.1-flash-lite'

print('Client ready. Prompts will go to', MODEL)

## Step 2: Ask a question

Run the cell below, type a prompt into the box that appears, and press Enter.

- `input` pauses the program until you press Enter and returns what you typed as a string.
- `generate_content` sends that string to the model and waits for its reply.
- `response.text` is the reply as an ordinary Python string.

In [ ]:
prompt = input('Please enter a prompt for the LLM:')

response = client.models.generate_content(model=MODEL, contents=prompt)

print(response.text)

Once the reply is in `response.text`, it is a string like any other from earlier in the semester. The rest of your program can print it, save it to a file, split it into words, or test it in an `if` statement.

## Try next

Each of these builds on Step 1, so run that cell first if you have restarted the notebook.

### Wrap the call in a function

With this function, asking an LLM a question looks the same as calling any other function we have written this semester.

In [ ]:
def ask_llm(prompt):
    """Send prompt to the LLM and return its reply as a string."""
    response = client.models.generate_content(model=MODEL, contents=prompt)
    return response.text

answer = ask_llm('In one sentence, what does a potentiometer measure?')
print(answer)

### Keep a conversation going

Each call to `generate_content` starts fresh, so the model does not see your earlier prompts. A chat object keeps the history and sends it along with every new message, which lets the model answer follow-up questions like "can you explain that more simply?"

Type `quit` to stop. Every message counts toward the free tier's per-minute limit, so if you see a `429` error, wait a minute.

In [ ]:
chat = client.chats.create(model=MODEL)

while True:
    prompt = input('You (type quit to stop): ')
    if prompt.strip().lower() == 'quit':
        break
    response = chat.send_message(prompt)
    print('LLM:', response.text)

### Give the model a job description

A **system instruction** is text the model reads before every prompt. It sets the model's role and the rules for its answers. Try changing it: have the model answer only in haiku, only with yes or no, or as a teaching assistant who refuses to write code, and see how the same question gets different answers.

In [ ]:
from google.genai import types

config = types.GenerateContentConfig(
    system_instruction='You are a patient teaching assistant for a first-year '
                       'engineering course. Answer in three sentences or fewer, '
                       'and explain ideas instead of writing code.'
)

prompt = input('Please enter a prompt for the LLM:')
response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
print(response.text)

## If something goes wrong

The [setup instructions](https://natrask.github.io/engr1050-fall2026/NewMaterial/LLM_API/) list the common errors and their fixes. The two you are most likely to see:

- **An error about `GEMINI_API_KEY`**: the secret's name must match exactly, and its **Notebook access** switch must be on.
- **`429 RESOURCE_EXHAUSTED`**: you sent more requests than the free tier allows. Wait a minute and try again. Do not upgrade to a paid tier.